In [ ]:
Update test.py; add to the end. Read the file fut_df_20230616_202309 and calculate volume imbalance.
Use my code style and my code templates

In [ ]:
Interview me relentlessly about every aspect of this plan until
we reach a shared understanding. Walk down each branch of the design
tree resolving dependencies between decisions one by one.

If a question can be answered by exploring the codebase, explore
the codebase instead.

For each question, provide your recommended answer.

In [ ]:
import numpy as np
import pandas as pd
# import plotly.graph_objects as go3
from datetime import datetime, timedelta
from numba import vectorize, float64, njit, jit, prange
# from skopt import Optimizer
import time
import gc
import datetime
import random
import os.path
import time
import shutil
from joblib import Parallel, delayed
import os
import itertools
# import holoviews as hv
from functools import reduce
# hv.extension('bokeh')

# conda activate 'your env'
# jupyter-lab --NotebookApp.iopub_data_rate_limit=1e10
pd.options.display.max_columns = 300
pd.options.display.max_rows = 10000
from math import inf

In [ ]:
file_types = 'parquet'
def remove_file(path):
    if os.path.isfile(path + '.csv'):
        os.remove(path + '.csv')
    elif os.path.isfile(path + '.parquet'):
        os.remove(path + '.parquet')
        
def read_file(path, read_columns=None):
    if file_types == 'csv' and read_columns is not None:
        return pd.read_csv(path + '.csv', sep=';', parse_dates=['ft_received'], engine='pyarrow', usecols=['ft_received', read_columns])
    elif file_types == 'csv':
        return pd.read_csv(path + '.csv', sep=';', engine='pyarrow')
    elif file_types == 'parquet' and read_columns is not None:
        return pd.read_parquet(path + '.parquet', engine='pyarrow', columns=read_columns)
    elif file_types == 'parquet':    
        return pd.read_parquet(path + '.parquet', engine='pyarrow')

def get_file_column_names(path):
    columns =[]
    if file_types == 'csv':
        df_header = pd.read_csv(path + '.csv', sep=';', nrows=0)
        columns = df_header.columns
    elif file_types == 'parquet':
        parquet_file = pq.ParquetFile(path + '.parquet')
        columns = parquet_file.schema.names
    return columns
    
def write_to_file(df, path):
    if file_types == 'csv':
        df.to_csv(path + '.csv', index=False, sep=';')
    elif file_types == 'parquet':
        df.to_parquet(path + '.parquet', engine='pyarrow', compression=None, index=False)

In [ ]:
def feature_engineering(idx):
    f_name = 'processed'
    remove_file('dataset//' + merged_files[idx] + ' ' + f_name)
    gc.collect()
    os.write(1, ('start ' + str(idx) + ' ' + str(time.time()) + '\n').encode())

    merged = pd.read_csv('merged_ds_'+merged_files[idx]+'.csv', sep=';', engine='pyarrow')
    merged.reset_index(inplace=True, drop=True)
    merged['ft_date'] = pd.to_datetime(merged['moment'].astype('str').str[:8], format='%Y%m%d').dt.strftime('%Y%m%d').astype('int')
    merged['ft_date_dt'] = pd.to_datetime(merged['moment'].astype('str').str[:8], format='%Y%m%d')
    merged['ft_time'] = merged['moment'].astype('str').str[8:14].astype('int')
    merged['ft_received'] = pd.to_datetime(merged['moment'].astype('str'), format='%Y%m%d%H%M%S%f')

    merged.drop(['pos_lst', 'moment'], axis=1, inplace=True)

    merged['second_counter'] = ((merged.ft_received - datetime.datetime(2021, 1, 1)) / np.timedelta64(1,'ms')).astype(np.int64)
    merged['month'] = merged.ft_received.dt.month
    merged['weekday'] = merged.ft_received.dt.weekday
    merged['day'] = merged.ft_received.dt.day
    merged['hour'] = merged.ft_received.dt.hour
    merged['minute'] = merged.ft_received.dt.minute
    merged['second'] = merged.ft_received.dt.second
    merged['part_of_day'] = pd.cut(merged.ft_received.dt.hour, bins=[8,14,19,24], labels=[0,1,2], include_lowest=True)

    write_to_file(merged, 'dataset//' + merged_files[idx] + ' ' + f_name)
    os.write(1, ('end ' + str(idx) + ' ' + str(time.time()) + '\n').encode())
    return 'OK ' + str(idx)

# for i in range(len(merged_files)):
#     feature_engineering(i)
res = Parallel(n_jobs=8, verbose=10)(delayed(feature_engineering)(i) for i in range(len(merged_files)))
print(res)

In [ ]:
def feature_engineering(idx):
    open_f_name = 'processed'
    new_f_name = 'processed2'

    os.write(1, ('start ' + str(idx) + ' ' + str(time.time()) + '\n').encode())
    remove_file('dataset//' + merged_files[idx] + ' ' + new_f_name)
    gc.collect()
    merged = read_file('dataset//' + merged_files[idx] + ' ' + open_f_name)
    merged.reset_index(inplace=True, drop=True)

    assert merged['second_counter'].dtype == np.int64
    assert merged['ft_price_ask'].dtype == 'int64'
    assert merged['ft_vol_ask'].dtype == 'int64'
    assert merged['ft_price_bid'].dtype == 'int64'
    assert merged['ft_vol_bid'].dtype == 'int64'
    assert merged['ft_vol_ask_traded_sum'].dtype == 'int64'
    assert merged['ft_vol_bid_traded_sum'].dtype == 'int64'
    assert merged['ft_price_ask_traded_sum'].dtype == 'int64'
    assert merged['ft_price_bid_traded_sum'].dtype == 'int64'
    assert merged['ft_count_ask_traded'].dtype == 'int64'
    assert merged['ft_count_bid_traded'].dtype == 'int64'
    assert merged['ft_price_ask_x_vol_ask_traded_sum'].dtype == 'int64'
    assert merged['ft_price_bid_x_vol_bid_traded_sum'].dtype == 'int64'
    ################################################
    shift_gap = '60s'
    merged['ft_price_ask_minus_shift'] = merged.ft_price_ask - merged.ft_price_ask.shift()
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_price_ask_minus_shift'] = np.nan
    merged['ft_price_bid_minus_shift'] = merged.ft_price_bid - merged.ft_price_bid.shift()
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_price_bid_minus_shift'] = np.nan
    merged['ft_price_ask_minus_price_bid'] = merged.ft_price_ask - merged.ft_price_bid
    merged['ft_price_ask_minus_price_bid_shift'] = merged.ft_price_ask_minus_price_bid - merged.ft_price_ask_minus_price_bid.shift()
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_price_ask_minus_price_bid_shift'] = np.nan
    merged['ft_avg'] = (merged.ft_price_ask + merged.ft_price_bid) / 2
    merged['ft_avg_shift'] = merged.ft_avg - merged.ft_avg.shift()
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_avg_shift'] = np.nan

    merged['ft_vol_ask_plus_ft_vol_bid'] = merged.ft_vol_ask + merged.ft_vol_bid
    merged['ft_vol_ask_plus_ft_vol_bid_shift'] = merged.ft_vol_ask_plus_ft_vol_bid - merged.ft_vol_ask_plus_ft_vol_bid.shift()
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_vol_ask_plus_ft_vol_bid_shift'] = np.nan
    merged['ft_vol_ask_minus_ft_vol_bid'] = merged.ft_vol_ask - merged.ft_vol_bid

    merged['ft_avg_x_ft_vol_ask_plus_ft_vol_bid'] = merged.ft_avg * merged.ft_vol_ask_plus_ft_vol_bid
    merged['ft_price_ask_x_vol_bid_plus_price_bid_x_vol_ask'] = merged.ft_price_ask * merged.ft_vol_bid + merged.ft_price_bid * merged.ft_vol_ask

    merged['microprice'] = round((merged.ft_price_ask * merged.ft_vol_bid + merged.ft_price_bid * merged.ft_vol_ask) / merged.ft_vol_ask_plus_ft_vol_bid, 1)
    merged['microprice_shift'] = round(merged.microprice - merged.microprice.shift(), 2)
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'microprice_shift'] = np.nan

    merged['volume_imbalance'] = np.round(merged.ft_vol_ask_minus_ft_vol_bid / merged.ft_vol_ask_plus_ft_vol_bid, 2)
    merged['volume_imbalance_shift'] = round(merged.volume_imbalance - merged.volume_imbalance.shift(), 2)
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'volume_imbalance_shift'] = np.nan

    merged['ft_price_ask_traded_sum_plus_ft_price_bid_traded_sum'] = (merged.ft_price_ask_traded_sum + merged.ft_price_bid_traded_sum)
    merged['ft_count_ask_traded_plus_ft_count_bid_traded'] = (merged.ft_count_ask_traded + merged.ft_count_bid_traded)
    merged['ft_avg_traded'] = round(merged['ft_price_ask_traded_sum_plus_ft_price_bid_traded_sum'] / merged['ft_count_ask_traded_plus_ft_count_bid_traded'].replace(0, np.nan), 1)
    merged['ft_avg_traded_shift'] = round(merged.ft_avg - merged.ft_avg.shift(), 2)
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'ft_avg_shift'] = np.nan

    merged['ft_price_ask_x_vol_ask_traded_sum_plus_ft_price_bid_x_vol_bid_traded_sum'] = (merged.ft_price_ask_x_vol_ask_traded_sum + merged.ft_price_bid_x_vol_bid_traded_sum)
    merged['ft_vol_ask_traded_sum_plus_ft_vol_bid_traded_sum'] = (merged.ft_vol_ask_traded_sum + merged.ft_vol_bid_traded_sum)
    merged['microprice_traded'] = round(merged['ft_price_ask_x_vol_ask_traded_sum_plus_ft_price_bid_x_vol_bid_traded_sum'] / merged['ft_vol_ask_traded_sum_plus_ft_vol_bid_traded_sum'].replace(0, np.nan), 1)
    merged['microprice_traded_shift'] = round(merged.microprice_traded - merged.microprice_traded.shift(), 2)
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'microprice_shift'] = np.nan

    merged['ft_vol_ask_traded_sum_minus_ft_vol_bid_traded_sum'] = (merged.ft_vol_ask_traded_sum - merged.ft_vol_bid_traded_sum)
    merged['ft_vol_ask_traded_sum_plus_ft_vol_bid_traded_sum'] = (merged.ft_vol_ask_traded_sum + merged.ft_vol_bid_traded_sum)
    merged['volume_imbalance_traded'] = round(merged['ft_vol_ask_traded_sum_minus_ft_vol_bid_traded_sum'] / merged['ft_vol_ask_traded_sum_plus_ft_vol_bid_traded_sum'].replace(0, np.nan), 2)
    merged['volume_imbalance_traded_shift'] = round(merged.volume_imbalance - merged.volume_imbalance.shift(), 2)
    merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, 'volume_imbalance_shift'] = np.nan

    # merged[''] = (merged.)
    # merged[''] = merged. - merged.
    # merged[''] = merged. / merged.
    # merged['_shift'] = merged. - merged..shift()
    # merged.loc[merged.ft_received - merged.ft_received.shift() > shift_gap, '_shift'] = np.nan
    ################################################
    write_to_file(merged, 'dataset//' + merged_files[idx] + ' ' + new_f_name)
    os.write(1, ('end ' + str(idx) + ' ' + str(time.time()) + '\n').encode())
    return 'OK ' + str(idx)

# for i in range(len(merged_files)):
#     feature_engineering(i)
res = Parallel(n_jobs=8, verbose=10)(delayed(feature_engineering)(i) for i in range(len(merged_files)))
print(res)

In [ ]:
def feature_engineering(col_values, merged_index, idx, tm, col_name, new_f_name, col_avg_values):    
    remove_file('dataset//' + merged_files[idx] + ' ' + new_f_name)
    remove_file('dataset//' + merged_files[idx] + ' temp ' + col_name + ' ' + tm + ' ' + new_f_name)
    gc.collect()
    os.write(1, ('start ' + str(idx) + ' ' + str(time.time()) + '\n').encode())

    temp = col_values.rolling(tm, min_periods=1).agg({
                                    # col_name + '_var_'+tm: 'var',
                                    col_name + '_std_'+tm: 'std',
                                    col_name + '_skew_'+tm: 'skew',
                                    col_name + '_kurt_'+tm: 'kurt',
                                    col_name + '_sem_'+tm: 'sem',
                                    col_name + '_mean_'+tm: 'mean',
                                    col_name + '_median_'+tm: 'median',
                                    col_name + '_max_'+tm: 'max',
                                    col_name + '_min_'+tm: 'min',
                                    col_name + '_open_minus_close_'+tm: lambda x: x.iloc[0] - x.iloc[-1],
                                    col_name + '_max_minus_min_'+tm: lambda x: np.max(x) - np.min(x),
                                    })
    temp[col_name + '_rank_'+tm] = col_values.rolling(tm, min_periods=5).rank(pct=True)
    
    rol_col_name = col_name + '_mean_'+tm
    temp[col_name+'_minus_'+rol_col_name] = col_values - temp[rol_col_name]
    rol_col_name = col_name + '_median_'+tm
    temp[col_name+'_minus_'+rol_col_name] = col_values - temp[rol_col_name]
    
    for temp_col_name in temp.columns:
        if col_name != 'ft_avg' and abs(np.mean(col_avg_values - temp[temp_col_name])) < 20:
            temp['ft_avg_minus_'+temp_col_name] = col_avg_values - temp[temp_col_name]
        
    # days_back = '5D'
    # for col_for_5D in temp.columns:
    #     temp[days_back+'_pct_'+col_for_5D] = temp[col_for_5D].rolling(days_back, min_periods=30000).rank(pct=True).to_frame(name='roll_5D')
    
    temp = temp.round(2)
    assert col_values.shape[0] == temp.shape[0] 
    temp.index = merged_index
    temp.to_csv('dataset//' + merged_files[idx] + ' temp ' + col_name + ' ' + tm + ' ' + new_f_name + '.csv', index=False, sep=';')
    os.write(1, ('end ' + str(idx) + '_' + tm + ' ' + str(time.time()) + ' ' + str(temp.shape[0]) + ' ' + str(col_values.shape[0]) + '\n').encode())

In [ ]:
col_name = 'ft_avg'
open_f_name = 'processed2'
new_f_name = 'processed3'
tms = ['3min', '1min', '30s', '15s', '6s', '3s', '1s']
merged_lst = []
merged_index_lst = []
for idx in range(len(merged_files)):
    df = read_file('dataset//' + merged_files[idx] + ' ' + open_f_name, read_columns=['ft_received', col_name])
    df.reset_index(inplace=True, drop=True)
    merged_index_lst.append(df.index)
    df.index = df.ft_received
    merged_lst.append(df)
Parallel(n_jobs=8, verbose=10)(delayed(feature_engineering)(merged_lst[param[0]][col_name], merged_index_lst[param[0]], param[0], param[1], col_name, new_f_name, merged_lst[param[0]]['ft_avg']) for param in itertools.product(list(range(len(merged_files))), tms))
# for param in itertools.product(list(range(len(merged_files))), tms):
#     feature_engineering(merged_lst[param[0]][col_name], merged_index_lst[param[0]], param[0], param[1], col_name, new_f_name, merged_lst[param[0]]['ft_avg'])

In [ ]:
@njit
def calculate_rolling_weighted_mean(second_counters, col_values, weights, tm, min_windows_length):
    n = len(second_counters)
    result = np.zeros(n)
    
    # Initialize window variables
    window_sum = 0.0
    weight_sum = 0.0
    window_count = 0
    start_idx = 0
    
    for i in range(n):
        current_time = second_counters[i]

        # Remove values outside the time window
        while start_idx < i and (current_time - second_counters[start_idx]) > tm:
            if col_values[start_idx] is not None and weights[start_idx] is not None:
                window_sum -= col_values[start_idx]
                weight_sum -= weights[start_idx]
                window_count -= 1  
            start_idx += 1

        # Add current value to window
        if col_values[i] is not None and weights[i] is not None:
            window_sum += col_values[i]
            weight_sum += weights[i]
            window_count += 1

        # Calculate weighted mean for current window
        if window_count > min_windows_length and abs(weight_sum) > 1e-10:
            result[i] = round(window_sum / weight_sum, 2)
        else:
            result[i] = np.nan
    return result

@njit
def calculate_rolling_slope_prediction(second_counters, col_values, tm, min_windows_length):
    MAX_SUM_X2 = 1e308  # Maximum value for double precision float
    n = len(second_counters)
    predicted_values = np.zeros(n)
    differences = np.zeros(n)

    last_time = second_counters[0]
    for i in range(n):
        assert last_time <= second_counters[i]
        last_time = second_counters[i]

    # Initialize window variables
    sum_x = 0.0  # sum of x values (time)
    sum_y = 0.0  # sum of y values (col_values)
    sum_xy = 0.0  # sum of x*y
    sum_x2 = 0.0  # sum of x^2
    window_count = 0
    start_idx = 0
    
    for i in range(n):
        current_time = second_counters[i]
        current_value = col_values[i]

        # Remove values outside the time window
        while start_idx < i and (current_time - second_counters[start_idx]) > tm:
            if col_values[start_idx] is not None:
                old_time = second_counters[start_idx]
                old_value = col_values[start_idx]
                sum_x -= old_time
                sum_y -= old_value
                sum_xy -= old_time * old_value
                sum_x2 -= old_time * old_time
                window_count -= 1
            start_idx += 1

        # Add current value to window
        if current_value is not None:
            sum_x += current_time
            sum_y += current_value
            sum_xy += current_time * current_value
            sum_x2 += current_time * current_time
            if sum_x2 > MAX_SUM_X2 or sum_xy > MAX_SUM_X2:
                raise OverflowError("sum_x2 overflow detected")
            window_count += 1

        # Calculate slope and intercept for current window
        if window_count >= min_windows_length and current_value is not None:
            denominator = window_count * sum_x2 - sum_x * sum_x
            if abs(denominator) > 1e-10:  # Avoid division by zero
                slope = (window_count * sum_xy - sum_x * sum_y) / denominator
                # Calculate intercept b
                b = (sum_y - slope * sum_x) / window_count
                # Calculate predicted value and difference
                predicted_value = round(slope * (current_time + tm) + b, 2)
                predicted_values[i] = predicted_value
                differences[i] = current_value - predicted_value
            else:
                predicted_values[i] = np.nan
                differences[i] = np.nan
        else:
            predicted_values[i] = np.nan
            differences[i] = np.nan
            
    return predicted_values, differences

@njit
def calculate_rolling_ewa(second_counters, col_values, beta, tm, min_windows_length):
# beta is a parameter (between 0 and 1) that determines the weight given to previous values
    n = len(second_counters)
    result = np.zeros(n)
    
    # Initialize EWA variables
    ewa_value = 0.0
    window_count = 0
    start_idx = 0
    
    for i in range(n):
        current_time = second_counters[i]
        current_value = col_values[i]

        # Remove values outside the time window
        while start_idx < i and (current_time - second_counters[start_idx]) > tm:
            if col_values[start_idx] is not None:
                window_count -= 1
            start_idx += 1

        # Add current value to window
        if current_value is not None:
            if window_count == 0:
                ewa_value = current_value
            else:
                ewa_value = beta * ewa_value + (1 - beta) * current_value
            window_count += 1

        # Calculate EWA for current window
        if window_count > min_windows_length:
            result[i] = round(ewa_value, 2)
        else:
            result[i] = np.nan
    return result

def feature_engineering_weight_means(merged, merged_index, idx, tm, col_name, weight_col_mame, new_f_name):
    remove_file('dataset//' + merged_files[idx] + ' ' + new_f_name)
    remove_file('dataset//' + merged_files[idx] + ' temp extra ' + col_name + ' ' + str(tm) + ' ' + new_f_name) 
    gc.collect()
    os.write(1, ('start ' + str(idx) + ' ' + str(time.time()) + '\n').encode())
    temp = pd.DataFrame()
    
    weight_means = calculate_rolling_weighted_mean(merged.second_counter.values, merged[col_name].values, merged[weight_col_mame].values, tm, min_windows_length=2)
    rol_col_name = col_name + '_del_' + weight_col_mame + '_weight_mean_' + str(tm)
    temp[rol_col_name] = weight_means
    temp[rol_col_name +'_minus_'+ col_name] = temp[rol_col_name] - merged[col_name] / merged[weight_col_mame]
    
    for temp_col_name in temp.columns:
        if col_name != 'ft_avg' and abs(np.mean(merged['ft_avg'] - temp[temp_col_name])) < 50:
            temp['ft_avg_minus_'+temp_col_name] = merged['ft_avg'] - temp[temp_col_name]
    
    # days_back = '5D'
    # for col_for_5D in temp.columns:
    #     temp_5d = pd.DataFrame()
    #     temp_5d[col_for_5D] = temp[col_for_5D].values
    #     temp_5d.index = merged.ft_received
    #     temp[days_back+'_pct_'+col_for_5D] = temp_5d[col_for_5D].rolling(days_back, min_periods=30000).rank(pct=True).to_frame(name='roll_5D').values
    
    temp = temp.round(2)
    assert merged.shape[0] == temp.shape[0] 
    temp.index = merged_index
    temp.to_csv('dataset//' + merged_files[idx] + ' temp extra ' + col_name + ' ' + str(tm) + ' ' + new_f_name + '.csv', index=False, sep=';')
    os.write(1, ('end ' + str(idx) + '_' + str(tm) + ' ' + str(time.time()) + ' ' + str(temp.shape[0]) + ' ' + str(merged.shape[0]) + '\n').encode())
    
def feature_engineering_ewa_and_slope(merged, merged_index, idx, tm, col_name, new_f_name):
    remove_file('dataset//' + merged_files[idx] + ' ' + new_f_name)
    remove_file('dataset//' + merged_files[idx] + ' temp extra ' + col_name + ' ' + str(tm) + ' ' + new_f_name) 
    gc.collect()
    os.write(1, ('start ' + str(idx) + ' ' + str(time.time()) + '\n').encode())
    temp = pd.DataFrame()
    
    test_predicted_values_by_slope, test_differences = calculate_rolling_slope_prediction([3, 3, 5, 9], [8, 2, 4, 6], 10, min_windows_length=4)    
    assert test_predicted_values_by_slope[-1] == 7.33 and test_differences[-1] == -1.33
    
    predicted_values_by_slope, differences = calculate_rolling_slope_prediction(merged.second_counter.values, merged[col_name].values, tm, min_windows_length=2)
    rol_col_name = col_name + '_predicted_by_slope_' + str(tm)
    temp[rol_col_name] = predicted_values_by_slope
    rol_col_name = col_name + '_minus_predicted_by_slope_' + str(tm)
    temp[rol_col_name] = differences
    